# Data Cleaning —

**Urban Pulse: Smart City Mobility Intelligence Platform**

Upload these 3 files to Colab first:
- `bangalore_integrated_mobility_dataset.csv`
- `Bangalore_Demographic_dataset_cleaned.xlsx`
- `mobility_clean.csv`

This checks for duplicates, nulls, bad data types, out-of-range values, 
and internal consistency (e.g. do percentages actually sum to 100%).

In [2]:
# DATA CLEANING — run BEFORE the merge step
# Urban Pulse - Smart City Mobility Intelligence Platform

In [ ]:
import pandas as pd
import numpy as np

In [4]:
# 1. CLEAN: bangalore_integrated_mobility_dataset.csv (main traffic+weather file)

In [5]:
traffic = pd.read_csv("bangalore_integrated_mobility_dataset.csv")
print("=== TRAFFIC + WEATHER FILE ===")
print(f"Starting shape: {traffic.shape}")

# --- Duplicates ---
dupes = traffic.duplicated().sum()
traffic = traffic.drop_duplicates()
print(f"Exact duplicate rows removed: {dupes}")

# --- Nulls ---
null_counts = traffic.isnull().sum()
null_cols = null_counts[null_counts > 0]
print(f"Columns with nulls:\n{null_cols if len(null_cols) else 'None'}")

# --- Data types ---
traffic["Date"] = pd.to_datetime(traffic["Date"], errors="coerce")
bad_dates = traffic["Date"].isna().sum()
print(f"Rows with unparseable Date: {bad_dates}")

# --- Strip whitespace from text columns ---
text_cols = traffic.select_dtypes(include="object").columns
for c in text_cols:
    traffic[c] = traffic[c].astype(str).str.strip()

# --- Outlier checks on key numeric columns (impossible values, not just extremes) ---
checks = {
    "Traffic Volume": (0, None),          # can't be negative
    "Average Speed": (0, 150),            # sanity ceiling
    "Congestion Level": (0, 100),         # it's a 0-100 score
    "Road Capacity Utilization": (0, 100),# it's a percentage
    "Traffic Signal Compliance": (0, 100),
    "Parking Usage": (0, 100),
}
for col, (lo, hi) in checks.items():
    if col in traffic.columns:
        bad = traffic[(traffic[col] < lo) | (hi is not None and traffic[col] > hi)]
        print(f"'{col}': {len(bad)} rows outside expected range [{lo}, {hi}]")

# --- Consistency check: Year/Month/Day columns vs parsed Date ---
mismatch = (traffic["Date"].dt.year != traffic["Year"]).sum()
print(f"Rows where Year column disagrees with parsed Date: {mismatch}")

print(f"Final shape: {traffic.shape}\n")

=== TRAFFIC + WEATHER FILE ===
Starting shape: (8936, 38)
Exact duplicate rows removed: 0
Columns with nulls:
None
Rows with unparseable Date: 0
'Traffic Volume': 0 rows outside expected range [0, None]
'Average Speed': 0 rows outside expected range [0, 150]
'Congestion Level': 0 rows outside expected range [0, 100]
'Road Capacity Utilization': 0 rows outside expected range [0, 100]
'Traffic Signal Compliance': 0 rows outside expected range [0, 100]
'Parking Usage': 0 rows outside expected range [0, 100]
Rows where Year column disagrees with parsed Date: 0
Final shape: (8936, 38)



In [6]:
# 2. CLEAN: Bangalore_Demographic_dataset_cleaned.xlsx

In [10]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------


[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: C:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [11]:
demo = pd.read_excel(r"c:\infosys internship task\given data\Bangalore_Demographic_dataset_cleaned (2).xlsx")
print("=== DEMOGRAPHIC FILE ===")
print(f"Starting shape: {demo.shape}")

dupes = demo.duplicated().sum()
demo = demo.drop_duplicates()
print(f"Exact duplicate rows removed: {dupes}")

null_counts = demo.isnull().sum()
null_cols = null_counts[null_counts > 0]
print(f"Columns with nulls:\n{null_cols if len(null_cols) else 'None'}")

# --- Consistency check: workers + non_workers should equal total_population ---
demo["_worker_check"] = demo["total_workers_persons"] + demo["non_workers_persons"]
mismatch = (demo["_worker_check"] != demo["total_population_persons"]).sum()
print(f"Rows where workers+non_workers != total_population: {mismatch}")
demo = demo.drop(columns=["_worker_check"])

# --- Strip whitespace from text columns ---
for c in demo.select_dtypes(include="object").columns:
    demo[c] = demo[c].astype(str).str.strip()

print(f"Final shape: {demo.shape}\n")

=== DEMOGRAPHIC FILE ===
Starting shape: (872, 11)
Exact duplicate rows removed: 0
Columns with nulls:
None
Rows where workers+non_workers != total_population: 0
Final shape: (872, 11)



In [12]:
# 3. CLEAN: mobility_clean.csv (zone-level mode share data)

In [13]:
zone = pd.read_csv("mobility_clean.csv")
print("=== ZONE-LEVEL MOBILITY FILE ===")
print(f"Starting shape: {zone.shape}")

dupes = zone.duplicated().sum()
zone = zone.drop_duplicates()
print(f"Exact duplicate rows removed: {dupes}")

null_counts = zone.isnull().sum()
null_cols = null_counts[null_counts > 0]
print(f"Columns with nulls:\n{null_cols if len(null_cols) else 'None'}")

# --- Fix: percentage columns are stored as TEXT with a '%' sign (e.g. "14%"),
#     not as numbers. Power BI can't chart/sum these until converted. ---
pct_cols = [c for c in zone.columns if
            "Mode Share" in c or "share" in c.lower() or "Trip Duration" in c]
for c in pct_cols:
    zone[c] = zone[c].astype(str).str.replace("%", "", regex=False).astype(float)
print(f"Converted {len(pct_cols)} percentage columns from text to numeric: {pct_cols}")

# --- Consistency check: the 8 mode share columns should sum to ~100 per zone ---
mode_cols = [c for c in zone.columns if c.endswith("Mode Share") or c == "Slow moving vehicles share"]
zone["_mode_sum"] = zone[mode_cols].sum(axis=1)
off = zone[(zone["_mode_sum"] < 95) | (zone["_mode_sum"] > 105)]
print(f"Mode share columns found: {mode_cols}")
print(f"Zones where mode shares don't sum to ~100%: {len(off)}")
if len(off):
    print(off[["Zone", "_mode_sum"]])
zone = zone.drop(columns=["_mode_sum"])

print(f"Final shape: {zone.shape}\n")

=== ZONE-LEVEL MOBILITY FILE ===
Starting shape: (11, 20)
Exact duplicate rows removed: 0
Columns with nulls:
None
Converted 10 percentage columns from text to numeric: ['Walk- Mode Share', 'Bicycle- Mode Share', 'Taxi/Maxi Cab- Mode Share', 'Auto- Mode Share', 'Two wheeler- Mode Share', 'Car/Van- Mode Share', 'Public Transport- Mode Share', 'Slow moving vehicles share', 'Trip Duration less than 15 mins', 'Trip Duration less than 30 mins']
Mode share columns found: ['Walk- Mode Share', 'Bicycle- Mode Share', 'Taxi/Maxi Cab- Mode Share', 'Auto- Mode Share', 'Two wheeler- Mode Share', 'Car/Van- Mode Share', 'Public Transport- Mode Share', 'Slow moving vehicles share']
Zones where mode shares don't sum to ~100%: 0
Final shape: (11, 20)



In [14]:
# Save cleaned outputs

In [15]:
traffic.to_csv("traffic_weather_cleaned.csv", index=False)
demo.to_excel("demographic_cleaned_final.xlsx", index=False)
zone.to_csv("mobility_zone_cleaned.csv", index=False)

print("Saved: traffic_weather_cleaned.csv, demographic_cleaned_final.xlsx, "
      "mobility_zone_cleaned.csv")
print("\nThese 3 files (+ your already-cleaned GTFS transit files) are now")
print("ready for the Step 5 merge notebook.")

Saved: traffic_weather_cleaned.csv, demographic_cleaned_final.xlsx, mobility_zone_cleaned.csv

These 3 files (+ your already-cleaned GTFS transit files) are now
ready for the Step 5 merge notebook.


In [1]:
import pandas as pd
import os

os.chdir(r"c:\infosys internship task\given data")

# --- Traffic + Weather ---
traffic = pd.read_csv("bangalore_integrated_mobility_dataset.csv")
traffic = traffic.drop_duplicates()
traffic["Date"] = pd.to_datetime(traffic["Date"], errors="coerce")
for c in traffic.select_dtypes(include="object").columns:
    traffic[c] = traffic[c].astype(str).str.strip()
traffic.to_csv("traffic_weather_cleaned.csv", index=False)

# --- Demographic ---
demo = pd.read_excel("Bangalore_Demographic_dataset_cleaned (2).xlsx")
demo = demo.drop_duplicates()
for c in demo.select_dtypes(include="object").columns:
    demo[c] = demo[c].astype(str).str.strip()
demo.to_excel("demographic_cleaned_final.xlsx", index=False)

# --- Zone mobility (fix % text columns) ---
zone = pd.read_csv("mobility_clean (1).csv")
zone = zone.drop_duplicates()
pct_cols = [c for c in zone.columns if "Mode Share" in c or "share" in c.lower() or "Trip Duration" in c]
for c in pct_cols:
    zone[c] = zone[c].astype(str).str.replace("%", "", regex=False).astype(float)
zone.to_csv("mobility_zone_cleaned.csv", index=False)

print("Saved to:", os.getcwd())
print([f for f in os.listdir() if f.endswith((".csv", ".xlsx"))])

FileNotFoundError: [Errno 2] No such file or directory: 'bangalore_integrated_mobility_dataset.csv'